# Deep Dive: Faithfulness, Hallucination & Grounding

This notebook is a focused deep-dive on **faithfulness** and **hallucination detection** in RAG systems. It builds on the basics covered in Notebook 04.

### What We Cover
1. Hallucination taxonomy (intrinsic, extrinsic, fabrication)
2. Edge cases: partial hallucination, ambiguous claims, inference
3. RAGAS vs DeepEval faithfulness comparison
4. Strategies to reduce hallucination
5. Temperature effects on faithfulness
6. Building a hallucination detection pipeline

> **Prerequisite:** Notebook 04 covers basic Faithfulness and Hallucination metrics.

In [ ]:
import os
import json
import pandas as pd
import numpy as np
from dotenv import load_dotenv

load_dotenv(os.path.join("..", ".env"))
assert os.getenv("OPENAI_API_KEY"), "Set OPENAI_API_KEY in your .env file"

from openai import OpenAI

openai_client = OpenAI()
print("Environment loaded.")

-
## Part 1: Understanding Hallucination Types

LLM hallucinations in RAG systems fall into three categories:

### 1. Intrinsic Hallucination (Contradiction)
The model generates information that **directly contradicts** the provided context.

### 2. Extrinsic Hallucination (Unsupported Addition)
The model adds information that is **not present in the context** but does not directly contradict it. The information might even be factually correct in general, but it is not grounded in the retrieved evidence.

### 3. Fabrication (Invention)
The model **invents entirely fictional** information -- names, numbers, events that have no basis in reality or the context.

In [ ]:
# Let us create clear examples of each hallucination type

CONTEXT = [
    "Acme Corp offers a 30-day return policy on all products purchased through "
    "our website or retail stores. Items must be unused, in original packaging, "
    "with receipt. Refunds are processed within 5-7 business days. Shipping costs "
    "for returns are the customer's responsibility unless the item was defective.",
    "Acme Corp offers Standard Shipping (5-7 days, free over $50), Expedited Shipping "
    "(2-3 days, $12.99), and Overnight Shipping (next business day, $24.99 if ordered "
    "before 2 PM EST). Acme Corp ships internationally to over 50 countries.",
]

# Example 1: Intrinsic Hallucination (Contradicts context)
intrinsic_example = {
    "query": "What is Acme Corp's return policy?",
    "context": CONTEXT,
    "response": (
        "Acme Corp offers a 60-day return policy on all items. Items can be "
        "returned even if opened and used. Refunds are instant."
    ),
    "hallucination_type": "Intrinsic (Contradiction)",
    "explanation": (
        "The response says 60 days (context says 30), says items can be returned "
        "even if opened and used (context says unused, original packaging), and says "
        "refunds are instant (context says 5-7 business days). These directly contradict the source."
    ),
}

# Example 2: Extrinsic Hallucination (Adds unsupported info)
extrinsic_example = {
    "query": "Tell me about Acme Corp's shipping.",
    "context": CONTEXT,
    "response": (
        "Acme Corp offers Standard, Expedited, and Overnight shipping options. "
        "Standard shipping is free for orders over $50. Acme Corp also offers "
        "a subscription service for $9.99/month that provides free expedited shipping "
        "on all orders. They recently partnered with FedEx for faster delivery."
    ),
    "hallucination_type": "Extrinsic (Unsupported Addition)",
    "explanation": (
        "The first two sentences are correct per the context. But the '$9.99/month "
        "subscription' and 'FedEx partnership' are not mentioned anywhere in the context. "
        "The LLM added plausible-sounding but unsupported claims."
    ),
}

# Example 3: Fabrication (Invents entirely)
fabrication_example = {
    "query": "What awards has Acme Corp won?",
    "context": CONTEXT,
    "response": (
        "Acme Corp won the prestigious Retail Excellence Award in 2023 and was named "
        "Best E-Commerce Company by Forbes. Their CEO received the Ernst & Young "
        "Entrepreneur of the Year award in 2022."
    ),
    "hallucination_type": "Fabrication (Invention)",
    "explanation": (
        "The context contains zero information about awards. The model invented specific "
        "awards, years, and accolades that sound plausible but are entirely fabricated. "
        "This is the most dangerous type of hallucination."
    ),
}

# Also create a faithful (non-hallucinated) example for comparison
faithful_example = {
    "query": "Tell me about Acme Corp's return policy.",
    "context": CONTEXT,
    "response": (
        "Acme Corp offers a 30-day return policy on all products purchased through "
        "their website or retail stores. Items must be unused, in original packaging, "
        "and accompanied by the receipt. Refunds are processed within 5-7 business days."
    ),
    "hallucination_type": "None (Faithful)",
    "explanation": "Every claim in this response can be directly traced to the provided context.",
}

all_examples = [faithful_example, intrinsic_example, extrinsic_example, fabrication_example]

print("Hallucination Examples:")
print("=" * 80)
for ex in all_examples:
    print(f"\n--- {ex['hallucination_type']} ---")
    print(f"  Query: {ex['query']}")
    print(f"  Response: {ex['response']}")
    print(f"  Why: {ex['explanation']}")

---
## Edge Cases: Partial Hallucination and Ambiguous Claims

These are the hardest cases for faithfulness metrics to handle correctly.

### Edge Cases: Partial Hallucination and Ambiguous Claims

In [ ]:
# Edge case 1: Partial hallucination -- mostly correct with one wrong detail
partial_tc = LLMTestCase(
    input="Tell me about Acme Corp's return policy.",
    actual_output=(
        "Acme Corp offers a 30-day return policy on all products. "
        "Refunds are processed within 5-7 business days. "
        "Returns are always free of charge with prepaid shipping labels."
    ),
    retrieval_context=[
        "Acme Corp offers a 30-day return policy on all products. Refunds are processed "
        "within 5-7 business days. Shipping costs for returns are the responsibility of the customer."
    ],
)

faithfulness_metric.measure(partial_tc)
print("Edge Case 1: Partial Hallucination")
print(f"  Score: {faithfulness_metric.score:.3f}")
print(f"  Reason: {faithfulness_metric.reason}")

In [ ]:
# Edge case 2: Ambiguous claim -- could be interpreted as supported or not
ambiguous_tc = LLMTestCase(
    input="Is Acme Corp a large company?",
    actual_output=(
        "Acme Corp ships internationally to over 50 countries, which suggests it is a large e-commerce company."
    ),
    retrieval_context=[
        "Acme Corp ships internationally to over 50 countries."
    ],
)

faithfulness_metric.measure(ambiguous_tc)
print("Edge Case 2: Ambiguous Claim (subjective classification)")
print(f"  Score: {faithfulness_metric.score:.3f}")
print(f"  Reason: {faithfulness_metric.reason}")

In [ ]:
# Edge case 3: Correct inference vs hallucination
inference_tc = LLMTestCase(
    input="How much does it cost to ship two separate orders overnight?",
    actual_output=(
        "Overnight Shipping costs $24.99 per order. "
        "For two separate orders, that would be $49.98 total."
    ),
    retrieval_context=[
        "Overnight Shipping is available for $24.99 and delivers the next business day "
        "if ordered before 2 PM EST."
    ],
)

faithfulness_metric.measure(inference_tc)
print("Edge Case 3: Mathematical Inference")
print(f"  Score: {faithfulness_metric.score:.3f}")
print(f"  Reason: {faithfulness_metric.reason}")

-
## Part 3: RAGAS Faithfulness -- Detailed Walkthrough

Let us run RAGAS Faithfulness on the same examples and compare results with DeepEval.

In [ ]:
from ragas import evaluate, EvaluationDataset, SingleTurnSample
from ragas.metrics import Faithfulness
from ragas.llms import LangchainLLMWrapper
from langchain_openai import ChatOpenAI

ragas_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4o-mini", temperature=0))

# Create RAGAS samples from our hallucination examples
ragas_samples = []
labels = []

for ex in all_examples:
    sample = SingleTurnSample(
        user_input=ex["query"],
        response=ex["response"],
        reference="",  # Not needed for faithfulness
        retrieved_contexts=ex["context"],
    )
    ragas_samples.append(sample)
    labels.append(ex["hallucination_type"])

ragas_dataset = EvaluationDataset(samples=ragas_samples)

# Run RAGAS Faithfulness
ragas_faith_results = evaluate(
    dataset=ragas_dataset,
    metrics=[Faithfulness(llm=ragas_llm)],
)

ragas_faith_df = ragas_faith_results.to_pandas()

print("RAGAS Faithfulness Results:")
print("=" * 60)
for idx, (row, label) in enumerate(zip(ragas_faith_df.itertuples(), labels)):
    print(f"\n  [{label}]")
    print(f"  RAGAS Faithfulness Score: {row.faithfulness:.3f}")

In [ ]:
# Compare RAGAS vs DeepEval Faithfulness on the same examples
de_scores = []
for ex in all_examples:
    tc = LLMTestCase(
        input=ex["query"],
        actual_output=ex["response"],
        retrieval_context=ex["context"],
    )
    quiet_metric = FaithfulnessMetric(model="gpt-4o-mini", threshold=0.7)
    quiet_metric.measure(tc)
    de_scores.append(quiet_metric.score)

comparison_df = pd.DataFrame({
    "Type": labels,
    "RAGAS_Faithfulness": ragas_faith_df["faithfulness"].values,
    "DeepEval_Faithfulness": de_scores,
})

print("\nComparison: RAGAS vs DeepEval Faithfulness")
print("=" * 70)
print(comparison_df.to_string(index=False))

print("\nKey observations:")
print("- Both frameworks should assign high scores to faithful responses")
print("- Both should detect contradictions (intrinsic hallucination)")
print("- Scores may differ due to different prompting strategies")
print("- Fabrication should get the lowest scores from both")

-
## Part 5: Strategies to Reduce Hallucination

Now let us test concrete strategies for reducing hallucination and **measure the improvement** using our faithfulness metrics.

In [ ]:
# Test context and queries we will use across all strategies
TEST_CONTEXT = [
    "Acme Corp offers a 30-day return policy on all products purchased through "
    "our website or retail stores. Items must be unused, in original packaging, "
    "with receipt. Refunds are processed within 5-7 business days. Shipping costs "
    "for returns are the customer's responsibility unless the item was defective.",
    "Acme Corp offers Standard Shipping (5-7 days, free over $50), Expedited Shipping "
    "(2-3 days, $12.99), and Overnight Shipping (next business day, $24.99). "
    "Acme Corp ships internationally to over 50 countries.",
    "All Acme Corp branded products come with a 1-year limited warranty covering "
    "defects in materials and workmanship. Extended warranty plans (2-year and "
    "3-year) are available for an additional fee.",
]

TEST_QUERIES = [
    "What is Acme Corp's return policy and how long do refunds take?",
    "What warranty do Acme products come with?",
    "What shipping options does Acme Corp offer and how much do they cost?",
    "Does Acme Corp offer same-day delivery?",
    "How does Acme Corp compare to its competitors?",
]

print(f"Testing with {len(TEST_QUERIES)} queries across multiple strategies")

In [ ]:
# Strategy 1: Baseline -- minimal system prompt
def generate_baseline(query, contexts):
    context_str = "\n\n".join(contexts)
    messages = [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {query}"},
    ]
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, temperature=0.7,
    )
    return response.choices[0].message.content

# Strategy 2: Grounding prompt -- explicit instruction to stay grounded
def generate_grounded(query, contexts):
    context_str = "\n\n".join(contexts)
    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant. ONLY answer based on the provided context. "
            "Do NOT add any information that is not explicitly stated in the context. "
            "If the answer is not in the context, say 'This information is not available "
            "in the provided context.' Do not speculate or infer beyond what is stated."
        )},
        {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {query}"},
    ]
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, temperature=0.7,
    )
    return response.choices[0].message.content

# Strategy 3: Low temperature
def generate_low_temp(query, contexts):
    context_str = "\n\n".join(contexts)
    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant. Answer based on the provided context."
        )},
        {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {query}"},
    ]
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, temperature=0.0,
    )
    return response.choices[0].message.content

# Strategy 4: Structured context formatting
def generate_structured(query, contexts):
    context_str = ""
    for i, ctx in enumerate(contexts, 1):
        context_str += f"[Document {i}]: {ctx}\n\n"
    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant. Answer ONLY using the information in the "
            "numbered documents below. When you make a claim, indicate which document "
            "number supports it using [Doc N] notation."
        )},
        {"role": "user", "content": f"Documents:\n{context_str}\nQuestion: {query}"},
    ]
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, temperature=0.0,
    )
    return response.choices[0].message.content

# Strategy 5: Chain-of-thought with citations
def generate_cot_citations(query, contexts):
    context_str = "\n\n".join(contexts)
    messages = [
        {"role": "system", "content": (
            "You are a helpful assistant. Follow these steps:\n"
            "1. First, identify which parts of the context are relevant to the question\n"
            "2. Extract the specific facts from the context\n"
            "3. Formulate your answer using ONLY those facts\n"
            "4. If the question cannot be fully answered from the context, explicitly state what is missing\n"
            "Always cite the relevant context in your answer."
        )},
        {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {query}"},
    ]
    response = openai_client.chat.completions.create(
        model="gpt-4o-mini", messages=messages, temperature=0.0,
    )
    return response.choices[0].message.content

strategies = {
    "Baseline (minimal prompt, temp=0.7)": generate_baseline,
    "Grounding prompt (temp=0.7)": generate_grounded,
    "Low temperature (temp=0.0)": generate_low_temp,
    "Structured context + citations": generate_structured,
    "Chain-of-thought + citations": generate_cot_citations,
}

print(f"Defined {len(strategies)} strategies to compare")

In [ ]:
# Run each strategy and collect responses
strategy_responses = {}

for strategy_name, gen_fn in strategies.items():
    print(f"\nRunning strategy: {strategy_name}")
    responses = []
    for query in TEST_QUERIES:
        response = gen_fn(query, TEST_CONTEXT)
        responses.append(response)
        print(f"  Processed: {query[:50]}...")
    strategy_responses[strategy_name] = responses

print(f"\nGenerated responses for all {len(strategies)} strategies")

In [ ]:
# Measure faithfulness for each strategy
strategy_scores = {}

for strategy_name, responses in strategy_responses.items():
    print(f"\nEvaluating strategy: {strategy_name}")
    scores = []
    
    for query, response in zip(TEST_QUERIES, responses):
        tc = LLMTestCase(
            input=query,
            actual_output=response,
            retrieval_context=TEST_CONTEXT,
        )
        metric = FaithfulnessMetric(model="gpt-4o-mini", threshold=0.7)
        metric.measure(tc)
        scores.append(metric.score)
    
    strategy_scores[strategy_name] = {
        "scores": scores,
        "mean": np.mean(scores),
        "min": np.min(scores),
        "max": np.max(scores),
    }
    print(f"  Mean: {strategy_scores[strategy_name]['mean']:.3f}")
    print(f"  Min:  {strategy_scores[strategy_name]['min']:.3f}")
    print(f"  Max:  {strategy_scores[strategy_name]['max']:.3f}")

In [ ]:
# Display strategy comparison
strategy_df = pd.DataFrame({
    "Strategy": list(strategy_scores.keys()),
    "Mean Faithfulness": [s["mean"] for s in strategy_scores.values()],
    "Min": [s["min"] for s in strategy_scores.values()],
    "Max": [s["max"] for s in strategy_scores.values()],
}).sort_values("Mean Faithfulness", ascending=False)

print("Strategy Comparison Results:")
print("=" * 80)
print(strategy_df.to_string(index=False))

best_strategy = strategy_df.iloc[0]["Strategy"]
print(f"\nBest strategy: {best_strategy}")
print(f"Mean faithfulness: {strategy_df.iloc[0]['Mean Faithfulness']:.3f}")

In [ ]:
# Show per-query breakdown for top 2 strategies
top_strategies = strategy_df.head(2)["Strategy"].values

print("Per-Query Breakdown (Top 2 Strategies):")
print("=" * 80)

per_query_data = []
for query_idx, query in enumerate(TEST_QUERIES):
    row = {"Query": query[:50]}
    for strategy_name in top_strategies:
        short_name = strategy_name[:25]
        row[short_name] = strategy_scores[strategy_name]["scores"][query_idx]
    per_query_data.append(row)

per_query_df = pd.DataFrame(per_query_data)
print(per_query_df.round(3).to_string(index=False))

### Temperature Effect on Faithfulness

Let us specifically isolate the effect of temperature on faithfulness. Higher temperature increases creativity/randomness, which can lead to more hallucination.

In [ ]:
# Test faithfulness at different temperature values
temperatures = [0.0, 0.3, 0.7, 1.0, 1.5]
temp_scores = {}

for temp in temperatures:
    print(f"\nTesting temperature={temp}")
    scores = []
    
    for query in TEST_QUERIES:
        context_str = "\n\n".join(TEST_CONTEXT)
        messages = [
            {"role": "system", "content": "You are a helpful assistant. Answer based on the provided context."},
            {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {query}"},
        ]
        response = openai_client.chat.completions.create(
            model="gpt-4o-mini", messages=messages, temperature=temp,
        )
        answer = response.choices[0].message.content
        
        tc = LLMTestCase(
            input=query,
            actual_output=answer,
            retrieval_context=TEST_CONTEXT,
        )
        metric = FaithfulnessMetric(model="gpt-4o-mini", threshold=0.7)
        metric.measure(tc)
        scores.append(metric.score)
    
    temp_scores[temp] = {
        "mean": np.mean(scores),
        "scores": scores,
    }
    print(f"  Mean faithfulness: {temp_scores[temp]['mean']:.3f}")

# Display temperature comparison
temp_df = pd.DataFrame({
    "Temperature": list(temp_scores.keys()),
    "Mean Faithfulness": [s["mean"] for s in temp_scores.values()],
})

print("\nTemperature vs Faithfulness:")
print("=" * 40)
print(temp_df.to_string(index=False))

-
## Part 6: Building a Hallucination Detection Pipeline

In production, you want an automated system that:
1. Evaluates every RAG response for faithfulness
2. Flags suspicious outputs
3. Provides actionable information about what went wrong

Let us build this pipeline.

In [ ]:
from dataclasses import dataclass
from enum import Enum
from datetime import datetime


class RiskLevel(Enum):
    LOW = "LOW"
    MEDIUM = "MEDIUM"
    HIGH = "HIGH"
    CRITICAL = "CRITICAL"


@dataclass
class HallucinationReport:
    """Report from hallucination detection pipeline."""
    query: str
    response: str
    faithfulness_score: float
    hallucination_score: float
    risk_level: RiskLevel
    issues: list
    recommendation: str
    timestamp: str


class HallucinationDetector:
    """Production hallucination detection pipeline.
    
    Combines multiple metrics to provide a comprehensive assessment
    of response grounding and flag potential hallucinations.
    """
    
    def __init__(self, model="gpt-4o-mini", faithfulness_threshold=0.8, hallucination_threshold=0.5):
        self.faithfulness_metric = FaithfulnessMetric(
            model=model, threshold=faithfulness_threshold,
        )
        self.hallucination_metric = HallucinationMetric(
            model=model, threshold=hallucination_threshold,
        )
        self.faithfulness_threshold = faithfulness_threshold
        self.hallucination_threshold = hallucination_threshold
    
    def detect(self, query: str, response: str, retrieval_context: list, context: list = None) -> HallucinationReport:
        """Run hallucination detection on a single response."""
        if context is None:
            context = retrieval_context
        
        # Run FaithfulnessMetric
        faith_tc = LLMTestCase(
            input=query,
            actual_output=response,
            retrieval_context=retrieval_context,
        )
        self.faithfulness_metric.measure(faith_tc)
        faithfulness_score = self.faithfulness_metric.score
        faith_reason = self.faithfulness_metric.reason
        
        # Run HallucinationMetric
        halluc_tc = LLMTestCase(
            input=query,
            actual_output=response,
            context=context,
        )
        self.hallucination_metric.measure(halluc_tc)
        hallucination_score = self.hallucination_metric.score
        halluc_reason = self.hallucination_metric.reason
        
        # Determine risk level
        issues = []
        
        if faithfulness_score < 0.5:
            issues.append(f"Very low faithfulness ({faithfulness_score:.2f}): {faith_reason}")
        elif faithfulness_score < self.faithfulness_threshold:
            issues.append(f"Below-threshold faithfulness ({faithfulness_score:.2f}): {faith_reason}")
        
        if hallucination_score < self.hallucination_threshold:
            issues.append(f"Hallucination detected ({hallucination_score:.2f}): {halluc_reason}")
        
        # Risk classification
        if faithfulness_score >= 0.9 and hallucination_score >= 0.8:
            risk_level = RiskLevel.LOW
            recommendation = "Response appears well-grounded. Safe to serve."
        elif faithfulness_score >= 0.7 and hallucination_score >= 0.5:
            risk_level = RiskLevel.MEDIUM
            recommendation = "Response has minor grounding issues. Consider review before serving."
        elif faithfulness_score >= 0.5 or hallucination_score >= 0.3:
            risk_level = RiskLevel.HIGH
            recommendation = "Significant hallucination risk. Should be reviewed by a human."
        else:
            risk_level = RiskLevel.CRITICAL
            recommendation = "Critical hallucination detected. DO NOT serve this response."
        
        return HallucinationReport(
            query=query,
            response=response,
            faithfulness_score=faithfulness_score,
            hallucination_score=hallucination_score,
            risk_level=risk_level,
            issues=issues,
            recommendation=recommendation,
            timestamp=datetime.now().isoformat(),
        )


detector = HallucinationDetector()
print("HallucinationDetector initialized.")

In [ ]:
# Test the detection pipeline on various responses
test_responses = [
    {
        "query": "What is Acme Corp's return policy?",
        "response": "Acme Corp offers a 30-day return policy on all products. Items must be unused, in original packaging, with receipt. Refunds are processed within 5-7 business days.",
        "label": "Faithful",
    },
    {
        "query": "What is Acme Corp's return policy?",
        "response": "Acme Corp offers a 30-day return policy. They also offer an extended 60-day holiday return window in December.",
        "label": "Extrinsic hallucination",
    },
    {
        "query": "What is Acme Corp's return policy?",
        "response": "Acme Corp offers a 90-day return policy with free return shipping on all items.",
        "label": "Intrinsic hallucination",
    },
    {
        "query": "What awards has Acme Corp won?",
        "response": "Acme Corp won the 2024 Retail Innovation Award and the National Customer Service Excellence recognition.",
        "label": "Fabrication",
    },
]

reports = []
for test in test_responses:
    print(f"\nAnalyzing: [{test['label']}] {test['query']}")
    report = detector.detect(
        query=test["query"],
        response=test["response"],
        retrieval_context=TEST_CONTEXT,
    )
    reports.append(report)
    print(f"  Risk Level: {report.risk_level.value}")
    print(f"  Faithfulness: {report.faithfulness_score:.3f}")
    print(f"  Hallucination: {report.hallucination_score:.3f}")
    print(f"  Recommendation: {report.recommendation}")
    if report.issues:
        for issue in report.issues:
            print(f"  Issue: {issue[:100]}...")

In [ ]:
# Summary of detection results
detection_df = pd.DataFrame({
    "Query": [r.query[:40] for r in reports],
    "Risk Level": [r.risk_level.value for r in reports],
    "Faithfulness": [r.faithfulness_score for r in reports],
    "Hallucination": [r.hallucination_score for r in reports],
    "Issues": [len(r.issues) for r in reports],
})

print("Detection Pipeline Summary:")
print("=" * 80)
print(detection_df.to_string(index=False))

# Statistics
risk_counts = detection_df["Risk Level"].value_counts()
print(f"\nRisk Level Distribution:")
for level, count in risk_counts.items():
    print(f"  {level}: {count}")

In [ ]:
# Building an alert system
class HallucinationAlertSystem:
    """Alert system that monitors RAG responses and raises alerts."""
    
    def __init__(self, detector: HallucinationDetector):
        self.detector = detector
        self.alerts = []
        self.total_checked = 0
        self.flagged_count = 0
    
    def check_response(self, query: str, response: str, retrieval_context: list) -> dict:
        """Check a single response and raise alert if needed."""
        self.total_checked += 1
        
        report = self.detector.detect(
            query=query,
            response=response,
            retrieval_context=retrieval_context,
        )
        
        result = {
            "query": query,
            "response": response,
            "risk_level": report.risk_level.value,
            "should_serve": report.risk_level in [RiskLevel.LOW, RiskLevel.MEDIUM],
            "needs_review": report.risk_level in [RiskLevel.HIGH, RiskLevel.CRITICAL],
        }
        
        if report.risk_level in [RiskLevel.HIGH, RiskLevel.CRITICAL]:
            self.flagged_count += 1
            alert = {
                "timestamp": report.timestamp,
                "risk_level": report.risk_level.value,
                "query": query,
                "faithfulness_score": report.faithfulness_score,
                "issues": report.issues,
                "recommendation": report.recommendation,
            }
            self.alerts.append(alert)
            print(f"  ALERT [{report.risk_level.value}]: {query[:50]}")
        
        return result
    
    def get_statistics(self) -> dict:
        """Get monitoring statistics."""
        return {
            "total_checked": self.total_checked,
            "flagged_count": self.flagged_count,
            "flag_rate": self.flagged_count / max(1, self.total_checked),
            "total_alerts": len(self.alerts),
        }


# Demo the alert system
alert_system = HallucinationAlertSystem(detector)

for test in test_responses:
    result = alert_system.check_response(
        query=test["query"],
        response=test["response"],
        retrieval_context=TEST_CONTEXT,
    )

stats = alert_system.get_statistics()
print(f"\nMonitoring Statistics:")
print(f"  Total responses checked: {stats['total_checked']}")
print(f"  Flagged responses: {stats['flagged_count']}")
print(f"  Flag rate: {stats['flag_rate']:.1%}")
print(f"  Total alerts: {stats['total_alerts']}")

In [ ]:
# Display all alerts
if alert_system.alerts:
    print("Active Alerts:")
    print("=" * 80)
    for i, alert in enumerate(alert_system.alerts, 1):
        print(f"\n  Alert #{i}")
        print(f"  Risk Level: {alert['risk_level']}")
        print(f"  Query: {alert['query']}")
        print(f"  Faithfulness: {alert['faithfulness_score']:.3f}")
        print(f"  Recommendation: {alert['recommendation']}")
        for issue in alert['issues']:
            print(f"  Issue: {issue[:100]}")
else:
    print("No alerts raised.")

-
## Summary

### Key Takeaways

1. **Hallucination types matter**: Intrinsic (contradiction) is most dangerous, extrinsic (unsupported) is common, fabrication is the worst case

2. **FaithfulnessMetric vs HallucinationMetric**: Use Faithfulness for strict RAG grounding (penalizes unsupported claims), use Hallucination for catching contradictions only

3. **RAGAS vs DeepEval Faithfulness**: Both use claim extraction + verification but with different prompting strategies. Scores may vary -- use both for comprehensive coverage.

4. **Reducing hallucination**: The most effective strategies are:
   - Strong grounding instructions in the system prompt
   - Low temperature (0.0-0.3)
   - Structured context with citation requirements
   - Chain-of-thought reasoning

5. **Production pipeline**: Combine multiple metrics, classify risk levels, and set up automated alerts for high-risk responses.

### Recommended Production Setup

- Set faithfulness threshold >= 0.85 for production
- Use temperature 0.0 for factual RAG responses
- Always include grounding instructions in the system prompt
- Monitor faithfulness scores over time to catch degradation
- Flag and review any response below threshold before serving to users

**Next notebook**: Evaluating agentic RAG systems with tool use and multi-turn conversations.